# Stage 1 Walkthrough — State Schema & Intent Extraction

This notebook builds Stage 1 of the AI search agent **piece by piece**, in the
same order the production code (`src/search_agent/...`) is organized, so you
can run each cell, inspect the output, and understand *why* each piece exists
before it all gets bundled into the final module.

**What Stage 1 does:** takes a free-text customer query like
*"waterproof jacket for an October coastal wedding"* and turns it into a
structured object:

```json
{"category": "Outerwear", "weather_attribute": "waterproof", "occasion": "Wedding", ...}
```

**Roadmap for this notebook:**
1. Define the global `GraphState` (what data flows through the whole agent)
2. Define `QueryIntent` (Stage 1's structured output contract, via Pydantic)
3. Write the prompts that ask Claude to fill in that structure
4. Call Claude with forced tool-use so the response *is* that structure
5. Add validation + retry-on-failure
6. Wrap it all into a single reusable node function
7. Run it against a few sample queries and inspect the results

> No API key? No problem for most of this notebook — a "fake Claude" is used
> in the validation/retry sections so you can see the *mechanics* without
> spending API credits. The only cell that needs a real key is clearly marked.

## 0. Setup

Just the imports we'll need. Nothing here is search-agent-specific yet —
that starts in section 1.

In [17]:
import json
import operator
from typing import Any, Dict, List, Literal, Optional

from pydantic import BaseModel, ConfigDict, Field, ValidationError
from typing_extensions import Annotated, NotRequired, TypedDict

print("Imports OK")

Imports OK


## 1. The Global State (`GraphState`)

Every node in a LangGraph pipeline reads from and writes to one shared
**state** object. Instead of discovering its shape by trial and error, we
define it up front as a `TypedDict` — a plain dict with a type-checked shape.

Why define *all four stages'* fields now, even though we're only building
Stage 1?  Because it means Stage 2/3/4 can be bolted on later **without**
changing Stage 1's node signature — every node just reads the keys it needs
and returns the keys it produces.

A few things worth noticing below:
- `raw_query` is the only field required at graph entry.
- Everything else is `NotRequired[...]` — it gets filled in stage by stage.
- `errors` uses `Annotated[List[str], operator.add]`. That annotation is a
  **reducer**: it tells LangGraph "if two parallel nodes both write to
  `errors` in the same step, concatenate the lists instead of one
  overwriting the other." We'll need that in Stage 2, where two nodes run
  in parallel.

In [18]:
class GraphState(TypedDict):
    # ---- Graph input (required at invocation) ----------------------------
    raw_query: str
    customer_id: NotRequired[Optional[str]]  # None => anonymous / guest session

    # ---- Stage 1: Intent Extraction --------------------------------------
    intent: NotRequired[Optional[Dict[str, Any]]]

    # ---- Stage 2: Candidate Retrieval & Profile Fetching (parallel) ------
    candidate_products: NotRequired[List[Dict[str, Any]]]      # Node 2A output
    customer_profile: NotRequired[Optional[Dict[str, Any]]]    # Node 2B output

    # ---- Stage 3: Relational Join & Personalization ----------------------
    filtered_skus: NotRequired[List[Dict[str, Any]]]

    # ---- Stage 4: Synthesis ------------------------------------------------
    final_response: NotRequired[Optional[str]]

    # ---- Cross-cutting ------------------------------------------------------
    errors: Annotated[List[str], operator.add]

print("GraphState defined")

GraphState defined


Let's instantiate a minimal state — the way the graph would look the instant
a user submits a query, before any node has run.

In [19]:
sample_state: GraphState = {
    "raw_query": "waterproof jacket for an October coastal wedding",
    "errors": [],
}
sample_state

{'raw_query': 'waterproof jacket for an October coastal wedding', 'errors': []}

## 2. The Intent Schema (`QueryIntent`)

This is the contract for what Stage 1 must produce. We use **Pydantic**
(not just a dict) because it gives us:
- Validation (a bad value raises immediately, loudly, instead of silently
  corrupting downstream logic)
- `Literal` enums we can constrain to the *actual* vocabulary used in the
  product catalog (so Stage 3's filtering can match on these fields exactly,
  with no fuzzy string matching)
- `.model_json_schema()` — a JSON Schema we can hand directly to Claude as a
  tool definition (see section 4). One schema, defined once, used for both
  "what do I ask Claude for" and "how do I validate what comes back."

Notice `category` / `occasion` / `weather_attribute` are all optional
`Literal[...]` types: if the query doesn't mention a wedding, `occasion`
should be `None`, not a guess.

In [20]:
Category = Literal["Outerwear", "Footwear", "Tops", "Bottoms", "Accessories"]
Occasion = Literal["Casual", "Formal", "Wedding", "Outdoor", "Activewear", "Business", "Travel"]
WeatherAttribute = Literal[
    "waterproof", "water-resistant", "weatherproof", "windproof", "insulated", "quick-drying"
]
print("Enums defined")

Enums defined


In [21]:
class QueryIntent(BaseModel):
    """Parsed representation of a free-text customer search query."""

    model_config = ConfigDict(extra="forbid")

    raw_query: str = Field(..., description="The original, unmodified user query.")

    category: Optional[Category] = Field(default=None)
    occasion: Optional[Occasion] = Field(default=None)
    weather_attribute: Optional[WeatherAttribute] = Field(default=None)
    material: Optional[str] = Field(default=None)
    size: Optional[str] = Field(default=None)
    color: Optional[str] = Field(default=None)
    max_price: Optional[float] = Field(default=None, ge=0)
    keywords: List[str] = Field(default_factory=list)

print("QueryIntent defined")

QueryIntent defined


### Try it: build a valid instance by hand

Before involving any LLM, let's just prove the schema itself behaves the way
we expect — this is exactly what will happen to Claude's tool-call output
once we validate it.

In [22]:
example_intent = QueryIntent(
    raw_query="waterproof jacket for an October coastal wedding",
    category="Outerwear",
    occasion="Wedding",
    weather_attribute="waterproof",
    keywords=["October", "coastal"],
)
example_intent.model_dump()

{'raw_query': 'waterproof jacket for an October coastal wedding',
 'category': 'Outerwear',
 'occasion': 'Wedding',
 'weather_attribute': 'waterproof',
 'material': None,
 'size': None,
 'color': None,
 'max_price': None,
 'keywords': ['October', 'coastal']}

### Try it: prove invalid values are rejected

`category="Suits"` isn't one of our allowed `Literal` values — this should
raise a `ValidationError` immediately, which is exactly the safety net we
want once real (occasionally sloppy) LLM output starts flowing through here.

In [23]:
try:
    QueryIntent(raw_query="test", category="Suits")
except ValidationError as e:
    print("Got expected ValidationError:\n")
    print(e)

Got expected ValidationError:

1 validation error for QueryIntent
category
  Input should be 'Outerwear', 'Footwear', 'Tops', 'Bottoms' or 'Accessories' [type=literal_error, input_value='Suits', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error


## 3. The Prompts

Prompts are kept as plain strings/functions, separate from any "node logic" —
so you (or a teammate) can tune wording without touching validation or
retry code.

- `INTENT_EXTRACTION_SYSTEM_PROMPT` sets the ground rules once.
- `build_intent_extraction_user_prompt(query)` just wraps the specific query.

In [24]:
INTENT_EXTRACTION_SYSTEM_PROMPT = """
You are the query-understanding component of a retail e-commerce search agent.

Your job is to read a customer's free-text product search query and extract a
structured representation of their intent by calling the `extract_query_intent` tool.

Rules:
- Only populate a field if the query actually states or strongly implies it.
  Leave a field empty/null rather than guessing.
- `category`, `occasion`, and `weather_attribute` MUST be one of the allowed
  enum values provided in the tool schema. If the query implies something close
  but not an exact match (e.g. "rain jacket" -> weather_attribute="waterproof"),
  map it to the closest allowed value.
- `keywords` is the catch-all for subjective, descriptive, or contextual terms
  that don't fit the structured fields (style adjectives, recipient, location,
  season, etc.) e.g. "stylish", "for my dad", "coastal", "October".
- `max_price` should be a plain number extracted from phrases like "under $150"
  or "around 80 dollars". Leave null if no price is mentioned.
- Always populate `raw_query` with the exact original query, unmodified.
- Call the tool exactly once with your best-effort structured extraction. Do not
  ask clarifying questions, partial information is expected and fine.
"""


def build_intent_extraction_user_prompt(raw_query: str) -> str:
    return f'Customer search query: "{raw_query}"\n\nExtract the structured intent.'


print(build_intent_extraction_user_prompt("waterproof jacket for an October coastal wedding"))

Customer search query: "waterproof jacket for an October coastal wedding"

Extract the structured intent.


## 4. Structured Output via Tool-Use

Here's the key idea: instead of asking Claude to *"please respond with
JSON"* (fragile, you end up regex-scraping code fences out of prose),
we give Claude a **tool** whose `input_schema` is generated directly from
`QueryIntent`, and force it to call that tool with `tool_choice`.

That means Claude's response literally contains a pre-parsed dict matching
our schema, we just hand it to Pydantic to validate.

Let's look at the schema we're about to hand over:

In [25]:
input_schema = QueryIntent.model_json_schema()
print(json.dumps(input_schema, indent=2))

{
  "additionalProperties": false,
  "description": "Parsed representation of a free-text customer search query.",
  "properties": {
    "raw_query": {
      "description": "The original, unmodified user query.",
      "title": "Raw Query",
      "type": "string"
    },
    "category": {
      "anyOf": [
        {
          "enum": [
            "Outerwear",
            "Footwear",
            "Tops",
            "Bottoms",
            "Accessories"
          ],
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Category"
    },
    "occasion": {
      "anyOf": [
        {
          "enum": [
            "Casual",
            "Formal",
            "Wedding",
            "Outdoor",
            "Activewear",
            "Business",
            "Travel"
          ],
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Occasion"
    }

In [26]:
tool_definition = {
    "name": "extract_query_intent",
    "description": "Return the structured search intent extracted from the customer's query.",
    "input_schema": input_schema,
}
tool_definition["name"], list(tool_definition["input_schema"]["properties"].keys())

('extract_query_intent',
 ['raw_query',
  'category',
  'occasion',
  'weather_attribute',
  'material',
  'size',
  'color',
  'max_price',
  'keywords'])

## 5. Calling the LLM (real API — needs `GROQ_API_KEY`)

This is the **one section** in the notebook that talks to the real API.
Set `GROQ_API_KEY` in your environment (or a `.env` file) before running
this cell. If it's not set, this cell will just print a message and skip;
the rest of the notebook works fine without it, using a fake client instead.

In [52]:
import os
from dotenv import load_dotenv

load_dotenv("/home/bhush/genai/recco/retail_search_agent/.env")  # loads .env if present

HAVE_API_KEY = bool(os.getenv("GROQ_API_KEY"))
print(f"GROQ_API_KEY set: {HAVE_API_KEY}")

GROQ_API_KEY set: True


In [55]:
from dotenv import load_dotenv

loaded = load_dotenv("/home/bhush/genai/recco/retail_search_agent/.env", override=True)

print(loaded)

True


In [57]:
if HAVE_API_KEY:
    from langchain_groq import ChatGroq
    from langchain_core.messages import HumanMessage, SystemMessage

    llm = ChatGroq(
        model=os.getenv("SEARCH_AGENT_LLM_MODEL", "llama-3.3-70b-versatile"),
        temperature=float(os.getenv("SEARCH_AGENT_TEMPERATURE", "0.7")),
        api_key=os.getenv("GROQ_API_KEY"),
    )

    # Use .with_structured_output() so the model returns a validated QueryIntent directly
    structured_llm = llm.with_structured_output(QueryIntent)

    real_parsed = structured_llm.invoke([
        SystemMessage(content=INTENT_EXTRACTION_SYSTEM_PROMPT),
        HumanMessage(content=build_intent_extraction_user_prompt(
            "waterproof jacket for an October coastal wedding"
        )),
    ])

    print("Structured output from Groq/Llama:")
    print(json.dumps(real_parsed.model_dump(), indent=2))
else:
    print("Skipping real API call, no GROQ_API_KEY found in the environment.")

Structured output from Groq/Llama:
{
  "raw_query": "waterproof jacket for an October coastal wedding",
  "category": "Outerwear",
  "occasion": "Wedding",
  "weather_attribute": "waterproof",
  "material": null,
  "size": null,
  "color": null,
  "max_price": null,
  "keywords": [
    "coastal",
    "October",
    "wedding"
  ]
}


## 6. A Fake Client, for Understanding the Mechanics Offline

The rest of this notebook uses a **fake client** that mimics a queued sequence
of canned responses (similar to what `GroqStructuredClient` in the production
code does) to exercise our validation and retry logic — no network, no API
key, fully deterministic.

This mirrors exactly what `tests/test_stage1_intent_extraction.py` does in
the full project.

In [ ]:
from types import SimpleNamespace


def make_tool_use_response(tool_name, tool_input):
    """Builds a fake response object shaped like a tool-use block."""
    block = SimpleNamespace(type="tool_use", name=tool_name, input=tool_input)
    return SimpleNamespace(content=[block])


class FakeMessages:
    """Stand-in for a chat model's message interface. Returns a queued
    sequence of canned responses, one per `.create()` call. This lets us
    simulate 'the model got it wrong once, then corrected itself.'"""

    def __init__(self, responses):
        self._responses = list(responses)
        self.call_count = 0

    def create(self, **kwargs):
        self.call_count += 1
        return self._responses.pop(0)


class FakeGroqClient:
    """Mimics a GroqStructuredClient for offline testing."""
    def __init__(self, responses):
        self.messages = FakeMessages(responses)


# Keep a backwards-compatible alias for older cells in this notebook
FakeAnthropicClient = FakeGroqClient


print("Fake client classes defined")

Fake client classes defined


### 6a. Happy path: valid output on the first try

In [ ]:
fake_client = FakeAnthropicClient([
    make_tool_use_response("extract_query_intent", {
        "raw_query": "waterproof jacket for an October coastal wedding",
        "category": "Outerwear",
        "occasion": "Wedding",
        "weather_attribute": "waterproof",
        "keywords": ["October", "coastal"],
    })
])

response = fake_client.messages.create()  # args don't matter for the fake
tool_use_block = next(b for b in response.content if b.type == "tool_use")
parsed = QueryIntent.model_validate(tool_use_block.input)
parsed.model_dump()

{'raw_query': 'waterproof jacket for an October coastal wedding',
 'category': 'Outerwear',
 'occasion': 'Wedding',
 'weather_attribute': 'waterproof',
 'material': None,
 'size': None,
 'color': None,
 'max_price': None,
 'keywords': ['October', 'coastal']}

### 6b. Validation-failure + retry path

Now let's simulate the model getting the `category` wrong on the first
attempt (`"Suits"` isn't a valid `Category`), and correcting itself on a
second attempt. This is the retry loop that `GroqStructuredClient` runs
in the full project, here it is unrolled by hand so you can see every step.

In [ ]:
retry_client = FakeAnthropicClient([
    make_tool_use_response("extract_query_intent", {
        "raw_query": "stylish blazer for a business dinner",
        "category": "Suits",       # invalid -> will fail validation
        "occasion": "Business",
    }),
    make_tool_use_response("extract_query_intent", {
        "raw_query": "stylish blazer for a business dinner",
        "category": "Tops",        # corrected
        "occasion": "Business",
        "keywords": ["stylish", "blazer"],
    }),
])

max_attempts = 2
result = None

for attempt in range(1, max_attempts + 1):
    response = retry_client.messages.create()
    tool_use_block = next(b for b in response.content if b.type == "tool_use")
    try:
        result = QueryIntent.model_validate(tool_use_block.input)
        print(f"Attempt {attempt}: validation succeeded")
        break
    except ValidationError as e:
        print(f"Attempt {attempt}: validation failed -> {e.errors()[0]['msg']}")
        # In the real client, the error text gets fed back to Claude here so
        # it can self-correct. We're simulating that correction by just
        # moving on to the next queued fake response.

result.model_dump() if result else None

Attempt 1: validation failed -> Input should be 'Outerwear', 'Footwear', 'Tops', 'Bottoms' or 'Accessories'
Attempt 2: validation succeeded


{'raw_query': 'stylish blazer for a business dinner',
 'category': 'Tops',
 'occasion': 'Business',
 'weather_attribute': None,
 'material': None,
 'size': None,
 'color': None,
 'max_price': None,
 'keywords': ['stylish', 'blazer']}

### 6c. Graceful fallback when every retry fails

If every attempt fails validation, Stage 1 should **never crash the graph**.
Instead it falls back to an "empty" `QueryIntent`, just `raw_query`
preserved, so Stage 2 can still fall back to plain keyword search.

In [ ]:
def empty_intent(raw_query):
    return QueryIntent(raw_query=raw_query).model_dump()


all_fail_client = FakeAnthropicClient([
    make_tool_use_response("extract_query_intent", {"raw_query": "x", "category": "NotReal"})
    for _ in range(3)
])

errors = []
parsed = None
for attempt in range(1, 4):
    response = all_fail_client.messages.create()
    tool_use_block = next(b for b in response.content if b.type == "tool_use")
    try:
        parsed = QueryIntent.model_validate(tool_use_block.input)
        break
    except ValidationError as e:
        errors.append(str(e))

if parsed is None:
    print("All attempts failed validation, falling back to empty intent")
    parsed_dict = empty_intent("some unparseable query")
else:
    parsed_dict = parsed.model_dump()

parsed_dict, f"{len(errors)} error(s) logged"

All attempts failed validation, falling back to empty intent

({'raw_query': 'some unparseable query',
  'category': None,
  'occasion': None,
  'weather_attribute': None,
  'material': None,
  'size': None,
  'color': None,
  'max_price': None,
  'keywords': []},
 '3 error(s) logged')

## 7. Putting It All Together: the LangGraph Node

Everything above, the schema, the prompts, the tool-use call, the
validate-and-retry loop, the graceful fallback, gets wrapped into **one**
function with the LangGraph node signature: `(state) -> partial state update`.

This is (a simplified, inline version of) `extract_intent_node` from
`src/search_agent/nodes/stage1_intent_extraction.py`.

In [58]:
def extract_intent_node(state, client=None, max_retries=2):
    """Stage 1 LangGraph node: state['raw_query'] -> {'intent': {...}, 'errors': [...]}.

    In the real project this defaults to a GroqStructuredClient backed by
    ChatGroq. Here we accept a fake client for offline demonstration.
    """
    raw_query = state["raw_query"]
    active_client = client  # in the real project defaults to GroqStructuredClient()

    messages = [{"role": "user", "content": build_intent_extraction_user_prompt(raw_query)}]
    last_error = None

    for attempt in range(1, max_retries + 2):
        # The fake client uses .messages.create(); the real GroqStructuredClient
        # uses ChatGroq.with_structured_output().invoke() — same semantics.
        response = active_client.messages.create(
            model="llama-3.3-70b-versatile",
            temperature=0.7,
            system=INTENT_EXTRACTION_SYSTEM_PROMPT,
            messages=messages,
            tools=[tool_definition],
        )
        tool_use_block = next((b for b in response.content if b.type == "tool_use"), None)
        if tool_use_block is None:
            last_error = "No tool_use block in response"
            continue
        try:
            intent = QueryIntent.model_validate(tool_use_block.input)
            return {"intent": intent.model_dump(), "errors": []}
        except ValidationError as e:
            last_error = str(e)

    return {
        "intent": empty_intent(raw_query),
        "errors": [f"stage1_intent_extraction: {last_error}"],
    }


print("extract_intent_node defined")

extract_intent_node defined


### Run the node against a few sample queries

Using the **fake client** so this runs offline. If you have a real Groq API
key set, swap the fake client for a real `GroqStructuredClient()`
(see section 5) to see live extraction on your own queries.

In [59]:
SAMPLE_QUERIES_AND_CANNED_RESPONSES = [
    (
        "waterproof jacket for an October coastal wedding",
        {
            "raw_query": "waterproof jacket for an October coastal wedding",
            "category": "Outerwear",
            "occasion": "Wedding",
            "weather_attribute": "waterproof",
            "keywords": ["October", "coastal"],
        },
    ),
    (
        "something stylish and breathable for the office, under $120",
        {
            "raw_query": "something stylish and breathable for the office, under $120",
            "category": "Tops",
            "occasion": "Business",
            "max_price": 120.0,
            "keywords": ["stylish", "breathable"],
        },
    ),
    (
        "durable size 10 hiking boots for the trail",
        {
            "raw_query": "durable size 10 hiking boots for the trail",
            "category": "Footwear",
            "occasion": "Outdoor",
            "size": "10",
            "keywords": ["durable", "hiking", "trail"],
        },
    ),
]

for query, canned_response in SAMPLE_QUERIES_AND_CANNED_RESPONSES:
    demo_client = FakeAnthropicClient([make_tool_use_response("extract_query_intent", canned_response)])
    state = {"raw_query": query, "errors": []}
    update = extract_intent_node(state, client=demo_client)

    print("=" * 80)
    print(f"QUERY: {query}")
    print("=" * 80)
    print(json.dumps(update["intent"], indent=2))
    print()

QUERY: waterproof jacket for an October coastal wedding
{
  "raw_query": "waterproof jacket for an October coastal wedding",
  "category": "Outerwear",
  "occasion": "Wedding",
  "weather_attribute": "waterproof",
  "material": null,
  "size": null,
  "color": null,
  "max_price": null,
  "keywords": [
    "October",
    "coastal"
  ]
}

QUERY: something stylish and breathable for the office, under $120
{
  "raw_query": "something stylish and breathable for the office, under $120",
  "category": "Tops",
  "occasion": "Business",
  "weather_attribute": null,
  "material": null,
  "size": null,
  "color": null,
  "max_price": 120.0,
  "keywords": [
    "stylish",
    "breathable"
  ]
}

QUERY: durable size 10 hiking boots for the trail
{
  "raw_query": "durable size 10 hiking boots for the trail",
  "category": "Footwear",
  "occasion": "Outdoor",
  "weather_attribute": null,
  "material": null,
  "size": "10",
  "color": null,
  "max_price": null,
  "keywords": [
    "durable",
    "hik

## 8. Recap & Next Steps

You just built, from the ground up:

1. **`GraphState`**, the shared state contract for the whole 4-stage agent
2. **`QueryIntent`**, Stage 1's structured output, with catalog-aligned enums
3. **Prompts**, kept separate from logic, easy to iterate on independently
4. **Forced tool-use**, one schema definition drives both "ask Claude" and
   "validate the answer"
5. **Validate + retry + graceful fallback**, a single bad LLM call degrades,
   it doesn't crash
6. **`extract_intent_node`**, the actual reusable LangGraph node

**Where this plugs into the full project:** everything in this notebook has
a 1:1 counterpart in `src/search_agent/` (see the project README), the
notebook is the "unrolled, cell-by-cell" version for learning; the package
is the "wired together, tested, importable" version for building on top of.

**Coming in Stage 2:** two nodes running in parallel:
- **2A** a lightweight custom TF-IDF / string-matching scorer over
  `products.csv`, using `state["intent"]` to fetch the top 15-20 candidates
- **2B** a CRM lookup against `customers.json` using `state["customer_id"]`

That's also where the `errors` reducer we defined back in section 1 earns
its keep, both nodes can log failures into the same `state["errors"]` list
without stepping on each other.